In [ ]:
# Verify and display the dataset folder structure
import os

# Base path for the dataset on Kaggle
BASE_DIR = "/kaggle/input/datasets/bhaveshmittal/melanoma-cancer-dataset"

# Fallback search in case the exact path differs slightly on Kaggle's mount point
if not os.path.exists(BASE_DIR):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train" in dirs and "test" in dirs:
            BASE_DIR = root
            break

print(f"Using dataset base directory: {BASE_DIR}\n")

def print_folder_structure(start_path, max_depth=2):
    for root, dirs, files in os.walk(start_path):
        depth = root.replace(start_path, "").count(os.sep)
        if depth > max_depth:
            continue
        indent = "    " * depth
        print(f"{indent}{os.path.basename(root) or root}/")
        if depth == max_depth:
            continue

print_folder_structure(BASE_DIR)

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

for split_dir, split_name in [(TRAIN_DIR, "Train"), (TEST_DIR, "Test")]:
    for cls in ["Benign", "Malignant"]:
        cls_path = os.path.join(split_dir, cls)
        if os.path.exists(cls_path):
            count = len(os.listdir(cls_path))
            print(f"{split_name} - {cls}: {count} images")


In [ ]:
# Core libraries
import os
import time
import random
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Scikit-learn for evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load training data and carve out a validation split
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.1,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.1,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Load held-out test data (never used during training)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Class Names:", class_names)
print(f"Training batches:   {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches:       {tf.data.experimental.cardinality(test_ds).numpy()}")


In [ ]:
# Count images per class in train and test folders
counts = {}
for split_dir, split_name in [(TRAIN_DIR, "Train"), (TEST_DIR, "Test")]:
    for cls in class_names:
        cls_path = os.path.join(split_dir, cls)
        counts[(split_name, cls)] = len(os.listdir(cls_path))

count_df = pd.DataFrame(
    [(s, c, n) for (s, c), n in counts.items()],
    columns=["Split", "Class", "Count"]
)
print(count_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - class distribution (train set)
train_counts = count_df[count_df["Split"] == "Train"]
sns.barplot(data=train_counts, x="Class", y="Count", hue="Class",
            palette="viridis", ax=axes[0], legend=False)
axes[0].set_title("Class Distribution (Training Set)")
axes[0].set_ylabel("Number of Images")

# Pie chart - class proportion (train set)
axes[1].pie(
    train_counts["Count"],
    labels=train_counts["Class"],
    autopct="%1.1f%%",
    colors=sns.color_palette("viridis", len(train_counts)),
    startangle=90
)
axes[1].set_title("Class Proportion (Training Set)")

plt.tight_layout()
plt.show()


In [ ]:
def show_random_samples(class_name, n=6):
    folder = os.path.join(TRAIN_DIR, class_name)
    files = random.sample(os.listdir(folder), n)
    fig, axes = plt.subplots(1, n, figsize=(18, 4))
    for ax, fname in zip(axes, files):
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")
    fig.suptitle(f"Random {class_name} Samples", fontsize=14)
    plt.tight_layout()
    plt.show()

show_random_samples("Benign")
show_random_samples("Malignant")


In [ ]:
def check_image_dimensions(class_name, n=50):
    folder = os.path.join(TRAIN_DIR, class_name)
    files = random.sample(os.listdir(folder), min(n, len(os.listdir(folder))))
    dims = []
    for fname in files:
        img = cv2.imread(os.path.join(folder, fname))
        dims.append(img.shape[:2])
    return dims

benign_dims = check_image_dimensions("Benign")
malignant_dims = check_image_dimensions("Malignant")

unique_dims = set(benign_dims + malignant_dims)
print("Unique image dimensions found in sample:", unique_dims)
print("All images match expected 224x224:", unique_dims == {(224, 224)})


In [ ]:
def get_pixel_values(class_name, n=30):
    folder = os.path.join(TRAIN_DIR, class_name)
    files = random.sample(os.listdir(folder), min(n, len(os.listdir(folder))))
    pixels = []
    for fname in files:
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pixels.append(img.reshape(-1, 3))
    return np.vstack(pixels)

benign_pixels = get_pixel_values("Benign")
malignant_pixels = get_pixel_values("Malignant")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
colors = ["red", "green", "blue"]
for i, color in enumerate(colors):
    axes[0].hist(benign_pixels[:, i], bins=50, alpha=0.5, color=color, label=color)
    axes[1].hist(malignant_pixels[:, i], bins=50, alpha=0.5, color=color, label=color)

axes[0].set_title("Pixel Intensity Distribution - Benign")
axes[1].set_title("Pixel Intensity Distribution - Malignant")
for ax in axes:
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Frequency")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Cache + prefetch pipelines for performance
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Confirm a sample batch shape and label encoding
for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Sample labels:", labels.numpy().flatten()[:10])
    print("0 ->", class_names[0], " | 1 ->", class_names[1])


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name="data_augmentation")

# Visualize augmented versions of a single sample image
for images, labels in train_ds.take(1):
    sample_image = images[0]
    break

plt.figure(figsize=(14, 6))
for i in range(8):
    augmented_image = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented_image[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Examples of Augmented Training Images", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
def build_model(input_shape=(224, 224, 3), base_trainable=False):
    inputs = layers.Input(shape=input_shape)

    # Augmentation only affects training; at inference it is a no-op
    x = data_augmentation(inputs)

    # EfficientNetB3 expects inputs in [0, 255]; preprocessing is built-in
    base_model = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = base_trainable

    x = base_model(x, training=base_trainable)
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs, outputs, name="EfficientNetB3_Melanoma_Classifier")
    return model, base_model

model, base_model = build_model(input_shape=(224, 224, 3), base_trainable=False)
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint("best_model_stage1.keras", monitor="val_auc", mode="max",
                     save_best_only=True, verbose=1)
]


In [ ]:
EPOCHS_STAGE1 = 30

start_time = time.time()

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks
)

stage1_training_time = time.time() - start_time
print(f"Stage 1 training time: {stage1_training_time:.2f} seconds")


In [ ]:
def plot_history(history, title_suffix=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["accuracy"], label="Training Accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title(f"Accuracy {title_suffix}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label="Training Loss")
    axes[1].plot(history.history["val_loss"], label="Validation Loss")
    axes[1].set_title(f"Loss {title_suffix}")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_stage1, title_suffix="(Stage 1 - Feature Extraction)")


In [ ]:
# Unfreeze the top layers of the backbone for fine-tuning
base_model.trainable = True

# Keep the first 70% of layers frozen; unfreeze the last 30%
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a much smaller learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

fine_tune_callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_model_finetuned.keras", monitor="val_auc", mode="max",
                     save_best_only=True, verbose=1)
]

EPOCHS_STAGE2 = 15

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=fine_tune_callbacks
)

total_training_time = stage1_training_time + (time.time() - start_time)


In [ ]:
plot_history(history_stage2, title_suffix="(Stage 2 - Fine-Tuning)")


In [ ]:
# Gather predictions on the test set
y_true, y_pred_prob = [], []

inference_start = time.time()
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_prob.extend(preds.flatten())
    y_true.extend(labels.numpy().flatten())
inference_time = time.time() - inference_start

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int)

print(f"Total inference time for {len(y_true)} test images: {inference_time:.2f} seconds")


In [ ]:
# Compute evaluation metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)  # Sensitivity
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_prob)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall (Sensitivity)", "Specificity",
               "F1 Score", "ROC AUC"],
    "Value": [acc, precision, recall, specificity, f1, roc_auc]
})
metrics_df["Value"] = metrics_df["Value"].apply(lambda x: f"{x:.4f}")
print(metrics_df.to_string(index=False))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()


In [ ]:
precisions, recalls, pr_thresholds = precision_recall_curve(y_true, y_pred_prob)

plt.figure(figsize=(7, 6))
plt.plot(recalls, precisions, color="purple", lw=2)
plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")


In [ ]:
# Collect test images and file-aligned predictions for visualization
all_images = []
for images, labels in test_ds:
    all_images.extend(images.numpy())
all_images = np.array(all_images)

assert len(all_images) == len(y_true), "Mismatch between collected images and labels"

sample_indices = random.sample(range(len(all_images)), 20)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, idx in zip(axes.flatten(), sample_indices):
    img = all_images[idx].astype("uint8")
    actual = class_names[int(y_true[idx])]
    predicted = class_names[int(y_pred[idx])]
    prob = y_pred_prob[idx]
    correct = actual == predicted

    ax.imshow(img)
    ax.axis("off")
    color = "green" if correct else "red"
    ax.set_title(f"Actual: {actual}\nPred: {predicted} ({prob:.2f})",
                 color=color, fontsize=10)

plt.suptitle("Prediction Gallery on Test Images", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"Total misclassified test images: {len(misclassified_idx)} / {len(y_true)}")

n_show = min(10, len(misclassified_idx))
show_idx = misclassified_idx[:n_show]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, idx in zip(axes.flatten(), show_idx):
    img = all_images[idx].astype("uint8")
    actual = class_names[int(y_true[idx])]
    predicted = class_names[int(y_pred[idx])]
    prob = y_pred_prob[idx]

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Actual: {actual}\nPred: {predicted} ({prob:.2f})",
                 color="red", fontsize=10)

plt.suptitle("Misclassified Test Images", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Explainability — LIME (replacement for Grad-CAM)
# Independent cell: does not modify or depend on any other
# cell's internals beyond reading existing variables.
# Required pre-existing variables: model, all_images, y_true,
# y_pred, class_names
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# --- 1. Ensure LIME is installed ---
try:
    import lime
    from lime import lime_image
except ImportError:
    import sys, subprocess
    print('LIME not found, installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lime'], check=True)
    import lime
    from lime import lime_image

from skimage.segmentation import mark_boundaries

# --- 2. Detect required variables from the existing notebook namespace ---
required_vars = ['model', 'all_images', 'y_true', 'y_pred', 'class_names']
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        f'Required variable(s) not found in the notebook namespace: {missing}. '
        f'Make sure Section 16 (Model Evaluation) and Section 20 (Prediction '
        f'Visualization) cells have been run first, since they define '
        f"'all_images', 'y_true', 'y_pred', and 'class_names'."
    )

print(f'Detected variables -> model: OK | all_images: {all_images.shape} | '
      f'y_true: {len(y_true)} samples | y_pred: {len(y_pred)} samples | '
      f'class_names: {class_names}')

# --- 3. Determine model output shape (single sigmoid vs 2-unit softmax) ---
n_output_units = model.output_shape[-1]
print(f"Model output units detected: {n_output_units} "
      f"({'sigmoid (binary)' if n_output_units == 1 else 'softmax (multi-unit)'})")

# --- 4. Prediction function wrapper for LIME ---
# LIME's image explainer requires a function that takes a batch of images
# (as float arrays in the model's native input range) and returns an
# (N, num_classes) array of class probabilities.
def predict_fn(images):
    # images: numpy array (N, H, W, 3), values in [0, 255], matching
    # how all_images is stored (EfficientNetB3 preprocessing is baked
    # into the model itself, so no extra normalization here).
    images = np.array(images, dtype=np.float32)

    if images.ndim == 3:
        images = np.expand_dims(images, axis=0)
    if images.shape[-1] == 4:
        images = images[..., :3]
    elif images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)

    preds = model.predict(images, verbose=0)
    preds = np.array(preds)
    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)

    if n_output_units == 1:
        p_malignant = preds.flatten()
        p_benign = 1.0 - p_malignant
        return np.stack([p_benign, p_malignant], axis=1)
    else:
        return preds

# --- 5. Sanity-check predict_fn on one image before running LIME ---
try:
    _test_pred = predict_fn(all_images[0:1])
    assert _test_pred.shape[1] == 2, (
        f'predict_fn must return shape (N, 2); got {_test_pred.shape}'
    )
except Exception as e:
    raise RuntimeError(f'predict_fn sanity check failed: {e}')

# --- 6. Select 6 random test images ---
n_explain = 6
if len(all_images) < n_explain:
    n_explain = len(all_images)
    print(f'Warning: fewer than 6 test images available; using {n_explain} instead.')

sample_lime_idx = random.sample(range(len(all_images)), n_explain)

# --- 7. Run LIME explainer on each sampled image ---
explainer = lime_image.LimeImageExplainer()

fig, axes = plt.subplots(n_explain, 2, figsize=(10, 4.5 * n_explain))
if n_explain == 1:
    axes = axes.reshape(1, 2)

for row, idx in enumerate(sample_lime_idx):
    try:
        img = all_images[idx].astype('uint8')
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.shape[-1] == 4:
            img = img[..., :3]

        true_label = class_names[int(y_true[idx])]
        pred_label = class_names[int(y_pred[idx])]
        confidence = predict_fn(np.expand_dims(img, axis=0))[0, int(y_pred[idx])]

        explanation = explainer.explain_instance(
            img.astype('double'),
            predict_fn,
            top_labels=2,
            hide_color=0,
            num_samples=1000,
            random_seed=SEED if 'SEED' in globals() else 42,
        )

        temp, mask = explanation.get_image_and_mask(
            label=int(y_pred[idx]),
            positive_only=True,
            num_features=8,
            hide_rest=False,
        )
        overlay = mark_boundaries(temp / 255.0, mask)

        axes[row, 0].imshow(img)
        axes[row, 0].axis('off')
        axes[row, 0].set_title(f'Original — True: {true_label}', fontsize=11)

        title_color = 'green' if pred_label == true_label else 'red'
        axes[row, 1].imshow(overlay)
        axes[row, 1].axis('off')
        axes[row, 1].set_title(
            f'LIME — Pred: {pred_label} ({confidence:.1%} confidence)',
            fontsize=11, color=title_color
        )

    except Exception as e:
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
        axes[row, 0].set_title(f'Error explaining sample idx={idx}', fontsize=10, color='red')
        print(f'Error generating LIME explanation for index {idx}: {e}')

plt.suptitle('LIME Explainability: Regions Supporting the Model\'s Prediction', fontsize=16, y=1.0)
plt.tight_layout()
plt.show()


In [ ]:
final_train_acc = history_stage2.history["accuracy"][-1]
final_val_acc = history_stage2.history["val_accuracy"][-1]

summary_df = pd.DataFrame({
    "Metric": [
        "Training Accuracy", "Validation Accuracy", "Test Accuracy",
        "Precision", "Recall (Sensitivity)", "Specificity", "F1 Score", "ROC AUC",
        "Total Training Time (s)", "Inference Time for Test Set (s)"
    ],
    "Value": [
        f"{final_train_acc:.4f}", f"{final_val_acc:.4f}", f"{acc:.4f}",
        f"{precision:.4f}", f"{recall:.4f}", f"{specificity:.4f}", f"{f1:.4f}", f"{roc_auc:.4f}",
        f"{total_training_time:.2f}", f"{inference_time:.2f}"
    ]
})

print(summary_df.to_string(index=False))
